In [29]:
import sys
import importlib

# Ensure utils are in path
from pathlib import Path
NOTEBOOK_DIR = Path.cwd()
UTILS_DIR = next(
    (candidate / "utils" for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (candidate / "utils").is_dir()),
    None,
)
if UTILS_DIR and str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

# Try to reload the modules
import two_stage_common
import regressor_tuning_common

importlib.reload(two_stage_common)
importlib.reload(regressor_tuning_common)

print("Modules reloaded.")
print(f"mlflow in two_stage_common: {two_stage_common.mlflow}")
print(f"mlflow in regressor_tuning_common: {regressor_tuning_common.mlflow}")

Modules reloaded.
mlflow in two_stage_common: <module 'mlflow' from 'c:\\Users\\PC\\OneDrive\\Área de Trabalho\\projetos\\.venv\\Lib\\site-packages\\mlflow\\__init__.py'>
mlflow in regressor_tuning_common: <module 'mlflow' from 'c:\\Users\\PC\\OneDrive\\Área de Trabalho\\projetos\\.venv\\Lib\\site-packages\\mlflow\\__init__.py'>


In [ ]:

# Investigating the Index/Columns dtype
from regressor_tuning_common import prepare_threshold_regression_data
import pandas as pd

data = prepare_threshold_regression_data(sample_frac=0.01)
X = data["X_search"]

print(f"Columns Index type: {type(X.columns)}")
print(f"Columns Index dtype: {X.columns.dtype}")
print(f"First 5 column names: {X.columns[:5].tolist()}")

# Check if any part of the index or metadata uses StringDtype
try:
    from pandas.api.types import is_string_dtype
    print(f"Is Columns Index string dtype? {is_string_dtype(X.columns)}")
except:
    pass

# Try to force columns to object
X.columns = X.columns.astype(object)
print(f"After conversion - Columns Index dtype: {X.columns.dtype}")

Columns Index type: <class 'pandas.Index'>
Columns Index dtype: str
First 5 column names: ['unidade', 'categoria_id', 'is_compravel', 'is_produzido_internamente', 'categoria']
Is Columns Index string dtype? True
After conversion - Columns Index dtype: object


# Tuning - Random Forest Regressor para limiar de reposicao

Este notebook desenvolve o sucessor do artefato historico `03_tree_ensembles_random_forest_regressor_threshold_model.pkl`, mas sem salvar `.pkl` localmente. O modelo, metricas, predicoes, parametros e graficos sao registrados diretamente no MLflow.


## Estrategia

Usamos `TimeSeriesSplit`, uma validacao cruzada temporal. Ela preserva a ordem passado -> futuro e evita vazamento temporal, que ocorreria com K-Fold aleatorio. O conjunto `test` fica isolado para avaliacao final do campeao.

O tuning usa `RandomizedSearchCV` porque o grid completo seria caro para florestas com varias combinacoes de profundidade, folhas e numero de arvores. Alem dos hiperparametros, cada busca e repetida com perfis de peso diferentes para testar custos de erro diferentes.


In [21]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
UTILS_DIR = next(
    (candidate / "utils" for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (candidate / "utils").is_dir()),
    None,
)
if UTILS_DIR is None:
    raise RuntimeError("Nao encontrei ml/notebooks/utils. Execute o notebook dentro do repositorio Saltim.")
sys.path.insert(0, str(UTILS_DIR))

import pandas as pd

from regressor_tuning_common import (
    RegressorTuningConfig,
    WEIGHT_PROFILE_DESCRIPTIONS,
    run_regressor_tuning,
)

from sklearn.ensemble import RandomForestRegressor


In [ ]:
RF_PARAM_DISTRIBUTIONS = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 6, 10, 16, 24],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", 0.5, 0.8, 1.0],
    "model__bootstrap": [True],
}

def rf_model_factory() -> RandomForestRegressor:
    return RandomForestRegressor(random_state=42, n_jobs=-1)

def rf_baseline_factory() -> RandomForestRegressor:
    return RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)

config = RegressorTuningConfig(
    notebook_id="05_random_forest_regressor_threshold_tuning",
    family="Arvores",
    model_name="Random Forest Regressor",
    model_factory=rf_model_factory,
    baseline_factory=rf_baseline_factory,
    param_distributions=RF_PARAM_DISTRIBUTIONS,
    n_iter=16,
    cv_splits=4,
    random_state=42,
    use_full_dataset=True,
)

search_space = pd.DataFrame(
    [{"parametro": key, "valores": values} for key, values in RF_PARAM_DISTRIBUTIONS.items()]
)
weight_profiles = pd.DataFrame(
    [{"perfil": key, "descricao": value} for key, value in WEIGHT_PROFILE_DESCRIPTIONS.items()]
)

display(search_space)
display(weight_profiles)


,parametro,valores
0,model__n_estimators,"[100, 200, 300, 500]"
1,model__max_depth,"[None, 6, 10, 16, 24]"
2,model__min_samples_split,"[2, 5, 10, 20]"
3,model__min_samples_leaf,"[1, 2, 4, 8]"
4,model__max_features,"[sqrt, log2, 0.5, 0.8, 1.0]"
5,model__bootstrap,[True]


,perfil,descricao
0,uniform,Peso 1 para todas as amostras; usado como cont...
1,alert_focus,Aumenta peso de observacoes em Alerta de compr...
2,critical_gap_focus,Aumenta peso quando cobertura esta abaixo/prox...
3,threshold_extreme_focus,Aumenta peso de limiares de alerta mais distan...


## Execucao do tuning

A celula abaixo executa baseline, buscas randomicas por perfil de peso, registro dos candidatos no MLflow, treino final do campeao e diagnosticos de ajuste. Se o tempo estiver alto, reduza `n_iter` mantendo os mesmos perfis de peso.


In [ ]:
results = run_regressor_tuning(config)

2026/06/01 16:11:16 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/06/01 16:11:17 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\PC\OneDrive\Área de Trabalho\projetos\.venv\Lib\site-packages\mlflow\data\digest_utils.py:29: Pandas4Warning: Starting with pandas version 4.0 all arguments of all will be keyword-only."
2026/06/01 16:11:21 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\PC\OneDrive\Área de Trabalho\projetos\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alt

🏃 View run Random Forest Regressor baseline at: http://localhost:5000/#/experiments/1/runs/3384acb456004dd29e40ac0731aa532e
🧪 View experiment at: http://localhost:5000/#/experiments/1


2026/06/01 16:58:01 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/06/01 16:58:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\PC\OneDrive\Área de Trabalho\projetos\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fitting 4 folds for each of 16 candidates, totalling 64 fits


## Resultados quantitativos

As tabelas abaixo mostram baseline, melhores candidatos, variancia entre folds e metricas finais no teste.


In [30]:
display(results["baseline_summary"])
display(results["candidate_results"].head(15))
display(results["best_fold_metrics"])
display(results["final_metrics"].T)
print("Melhor perfil de peso:", results["best_weight_profile"])
print("Melhores hiperparametros:", results["best_params"])
print("Diagnostico:", results["fit_diagnosis"])


NameError: name 'results' is not defined

## Visualizacoes de ajuste

A curva de aprendizado ajuda a identificar overfitting ou underfitting. A analise de residuos mostra vieses e dispersao dos erros no limiar previsto.


In [ ]:
results["figures"]["learning_curve"]


In [ ]:
results["figures"]["residual_analysis"]


## MLflow

No MLflow, procure pelos runs do experimento `notebooks/02_modelos_finais/05_random_forest_regressor_threshold_tuning/threshold_regression`. Os runs de candidatos guardam combinacoes de parametros e pesos; o run `champion` guarda o modelo registrado, predicoes, metricas finais e graficos.
